# Feature Engineering

The goal of this part is to design advanced functions to predict viral posts within the r/WallStreetBets dataset. Based on previous exploratory data analysis (EDA), a marked asymmetry in interaction was observed, with a small proportion of posts generating extreme outcomes. To address this, we framed the task as a rare event classification problem, defining viral posts as those in the top 5% of the score_log distribution.

Our function engineering strategy is designed to go beyond basic transformations and instead capture the structural, temporal, linguistic, and community-level dynamics that influence virality. In particular, we constructed continuous momentum functions, cyclical temporal encodings, structural content indicators, and nonlinear interaction terms to model the observed behavioral patterns on the subreddit.

We evaluated the impact of these designed functions using a logistic regression classifier and demonstrated that they substantially improve predictive performance.

In [52]:
# Feature Engineering (define viral posts)
import pandas as pd
import numpy as np

df = pd.read_csv("reddit_wsb_cleaned.csv")

# Define viral threshold (top 5%)
threshold = df["score_log"].quantile(0.95)
df["viral_flag"] = (df["score_log"] >= threshold).astype(int)

print(df["viral_flag"].value_counts(normalize=True))

viral_flag
0    0.949988
1    0.050012
Name: proportion, dtype: float64


In [53]:
# Additional features
df["body_present"] = df["body_clean"].notna().astype(int)
df["body_length"] = df["body_clean"].fillna("").str.len()
df["body_title_ratio"] = df["body_length"] / (df["title_length"] + 1)

In [54]:
# Time features
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["late_night_post"] = ((df["hour"] >= 21) | (df["hour"] <= 1)).astype(int)

In [55]:
# Community-level features
df["caps_ratio"] = df["title"].str.count(r"[A-Z]") / df["title"].str.len()
df["exclamation_count"] = df["title"].str.count("!")
df["question_count"] = df["title"].str.count(r"\?")
df["title_intensity"] = df["caps_ratio"] * (df["exclamation_count"] + 1)

In [56]:
# Specific time feature (rolling average of title length over past 50 posts)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

df["rolling_50_post_avg_length"] = (
    df["title_length"]
    .rolling(50, min_periods=1)
    .mean()
)

In [57]:
# Establish leakage columns to drop before modeling
leakage_cols = [
    "score",
    "score_log",
    "score_log_zscore",
    "score_log_minmax",
    "comms_num",
    "comms_num_log",
    "comms_num_log_zscore",
    "comms_num_log_minmax"
]

df = df.drop(columns=leakage_cols, errors="ignore")


In [58]:
# Check correlation of features with viral_flag
df.corr(numeric_only=True)["viral_flag"].sort_values(ascending=False)

viral_flag                    1.000000
hour                          0.088232
hour_zscore                   0.088232
hour_minmax                   0.088232
rolling_50_post_avg_length    0.088150
type_image                    0.029651
title_length                  0.015630
title_length_zscore           0.015630
title_length_minmax           0.015630
body_length                  -0.002450
exclamation_count            -0.016838
body_title_ratio             -0.024302
hour_sin                     -0.026106
late_night_post              -0.026751
title_intensity              -0.028913
day_of_week_encoded          -0.037083
caps_ratio                   -0.039084
question_count               -0.040395
hour_cos                     -0.073445
body_present                 -0.114075
has_body                     -0.114684
type_text                    -0.139748
Name: viral_flag, dtype: float64

In [59]:
# Time features (rolling count of posts in last 6 hours and last 50 posts)
df = df.sort_values("timestamp")

df["post_count_last_50"] = (
    df.index.to_series().rolling(50, min_periods=1).count()
)

df["posts_last_6h"] = (
    df.set_index("timestamp")
      .rolling("6h")
      .count()["id"]
      .values
)

In [60]:
# Interaction features (type of post (image/text) with late night posting)
df["image_late_night"] = df["type_image"] * df["late_night_post"]
df["text_late_night"] = df["type_text"] * df["late_night_post"]

In [61]:
# Text features (average word length, word count, unique word ratio)
df["avg_word_length"] = (
    df["title"].str.split().apply(lambda x: np.mean([len(word) for word in x]) if len(x) > 0 else 0)
)

df["word_count"] = df["title"].str.split().apply(len)
df["unique_word_ratio"] = (
    df["title"].str.split()
      .apply(lambda x: len(set(x)) / len(x) if len(x) > 0 else 0)
)

In [62]:
# r/WallStreetBets specific text features (rocket emoji, all caps words, price targets)
df["contains_rocket"] = df["title"].str.contains("🚀", regex=False).astype(int)
df["contains_all_caps_word"] = df["title"].str.contains(r"\b[A-Z]{4,}\b").astype(int)
df["contains_price_target"] = df["title"].str.contains(r"\$\d+", regex=True).astype(int)

In [63]:
# Interaction features (caps ratio with exclamation count, title length with caps ratio)
df["caps_exclaim_interaction"] = df["caps_ratio"] * df["exclamation_count"]
df["length_intensity_interaction"] = df["title_length"] * df["caps_ratio"]

In [64]:
# Check correlation of features with viral_flag
df.corr(numeric_only=True)["viral_flag"].sort_values(ascending=False)

viral_flag                      1.000000
hour                            0.088232
hour_zscore                     0.088232
hour_minmax                     0.088232
rolling_50_post_avg_length      0.088150
contains_price_target           0.032579
type_image                      0.029651
title_length_zscore             0.015630
title_length                    0.015630
title_length_minmax             0.015630
word_count                      0.012685
post_count_last_50              0.006064
contains_rocket                 0.004965
image_late_night                0.002034
unique_word_ratio               0.001775
avg_word_length                 0.000291
body_length                    -0.002450
caps_exclaim_interaction       -0.016138
exclamation_count              -0.016838
length_intensity_interaction   -0.020286
body_title_ratio               -0.024302
hour_sin                       -0.026106
late_night_post                -0.026751
title_intensity                -0.028913
day_of_week_enco

In [65]:
# Time-based baseline (average viral_flag by hour of day)
hour_baseline = (
    df.groupby("hour")["viral_flag"]
      .mean()
)

df["hour_viral_baseline"] = df["hour"].map(hour_baseline)

# Specific feature that measures how much above or below the hour baseline each post is
df["above_hour_baseline"] = (
    df["hour_viral_baseline"] - df["hour_viral_baseline"].mean()
)

In [66]:
# Type-based baseline (average viral_flag by type of post)
type_baseline = (
    df.groupby("type_image")["viral_flag"]
      .mean()
)

df["type_viral_baseline"] = df["type_image"].map(type_baseline)

In [67]:
# Time-based rolling feature (rolling average viral_flag over past 200 posts)
df = df.sort_values("timestamp")

df["rolling_200_viral_rate"] = (
    df["viral_flag"]
      .shift(1)
      .rolling(200, min_periods=20)
      .mean()
)

In [68]:
import math

# Text feature using log (Shannon entropy of title)
def shannon_entropy(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0
    prob = [text.count(c) / len(text) for c in set(text)]
    return -sum(p * math.log2(p) for p in prob)

df["title_entropy"] = df["title"].apply(shannon_entropy)

In [69]:
# Categorical feature engineering (very short vs very long titles)
title_95th = df["title_length"].quantile(0.95)

df["very_short_title"] = (df["title_length"] <= 10).astype(int)
df["very_long_title"] = (df["title_length"] >= title_95th).astype(int)

# Onto modeling...

In [70]:
# Drop original text columns and timestamp before modeling
drop_cols = [
    "title",
    "body",
    "body_clean",
    "title_clean",
    "title_nlp",
    "timestamp"
]

df_model = df.drop(columns=drop_cols, errors="ignore")

df_model = df_model.fillna(0)

In [71]:
# Prepare features and target
X = df_model.drop(columns=["viral_flag"])
y = df_model["viral_flag"]

In [72]:
# Keep only numeric columns
X = df_model.drop(columns=["viral_flag"])
y = df_model["viral_flag"]

X = X.select_dtypes(include=[np.number]).copy()

X = X.drop(columns=["id"], errors="ignore")

In [73]:
from sklearn.model_selection import train_test_split

# Split data into train and test sets
X = df_model.drop(columns=["viral_flag"])
X = X.select_dtypes(include=[np.number])
y = df_model["viral_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [74]:
from sklearn.preprocessing import StandardScaler

# Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [75]:
from sklearn.linear_model import LogisticRegression

# Train logistic regression model with class weighting to handle imbalance
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000)

In [76]:
from sklearn.metrics import roc_auc_score, classification_report

# Evaluate model and get AUC
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
print("AUC:", auc)

print(classification_report(y_test, model.predict(X_test_scaled)))

AUC: 0.8288597632017904
              precision    recall  f1-score   support

           0       0.98      0.72      0.83     10106
           1       0.13      0.78      0.22       532

    accuracy                           0.72     10638
   macro avg       0.56      0.75      0.53     10638
weighted avg       0.94      0.72      0.80     10638



In [77]:
import pandas as pd

# Investigate top features based on logistic regression coefficients
feature_importance = pd.Series(
    model.coef_[0],
    index=X.columns
).sort_values(ascending=False)

print("Top Positive Features:")
print(feature_importance.head(10))

print("\nTop Negative Features:")
print(feature_importance.tail(10))

Top Positive Features:
rolling_200_viral_rate        0.817257
post_count_last_50            0.813249
body_present                  0.544673
body_length                   0.403745
late_night_post               0.243814
hour_viral_baseline           0.145251
above_hour_baseline           0.145251
caps_ratio                    0.128454
rolling_50_post_avg_length    0.100239
hour_cos                      0.097901
dtype: float64

Top Negative Features:
avg_word_length          -0.097318
image_late_night         -0.177609
question_count           -0.182433
type_viral_baseline      -0.245442
contains_all_caps_word   -0.343357
word_count               -0.371966
posts_last_6h            -0.398902
text_late_night          -0.437823
body_title_ratio         -0.450204
has_body                 -1.228641
dtype: float64


# Feature Engineering Summary

## 1. Problem Framing

Given the heavy-tailed nature of engagement observed during EDA, we reframed the predictive task from modeling raw engagement counts to identifying viral posts, defined as those in the top 5% of the empirical score_log distribution.

This formulation:
- Addresses skewness in engagement
- Focuses on rare-event dynamics
- Aligns with observed extreme-value behavior in the dataset

Virality was posed as a binary system, where viral_flag = 1 if score_log is greater than or equal to Q_0.95, and 0 otherwise.

## 2. Strategy

The feature engineering was guided directly by our EDA findings. Rather than applying generic transformations, we constructed features designed to capture behavioral, temporal, structural, and community-level mechanisms driving virality.

### Structural Features

EDA revealed that over 53% of posts lacked body content. We hypothesized that post structure may influence virality.

Engineered features:
- body_present
- body_length
- body_title_ratio
- very_short_title
- very_long_title

### Time Encoding

EDA showed posting hour influenced engagement.

Instead of using raw hour, we applied cyclical encoding:
- hour_sin
- hour_cos- This preserves periodicity (e.g., hour 23 is close to hour 0).

We then introduced:
- late_night_post
- hour_viral_baseline
- above_hour_baseline

These features create both periodic structure and deviation from historical viral rates.

### Momentum Features

To model the wavelike quality of virality, we modeled:
- rolling_200_viral_rate (shifted to avoid leakage)
- post_count_last_50
- posts_last_6h

These features measure community activity intensity and prior viral frequency.

We got the rolling viral rate using .shift(1) to make sure no future information leaked into predictions.

### Text Intensity

Given the viral/hype culture of r/WallStreetBets, we engineered linguistic intensity metrics:
- caps_ratio
- exclamation_count
- title_intensity
- contains_rocket
- contains_all_caps_word
- title_entropy
- unique_word_ratio

### Interaction Features

We modeled nonlinear effects with:
- image_late_night
- caps_exclaim_interaction
- length_intensity_interaction

## 3. Model Validation

To evaluate whether engineered features enhanced predictive power, we trained a logistic regression model with class balancing.

Performance:
- AUC: 0.83
- Viral recall: 0.78

An AUC of 0.83 in a 5% rare-event classification task indicates strong discriminative ability.

The most influential features were:
- rolling_200_viral_rate
- post_count_last_50
- body_present
- late_night_post
- caps_ratio

## 4. Evidence of Enhanced Predictive Power

Compared to a baseline model using only basic metadata (e.g., hour, title length), the inclusion of engineered temporal and community-level features substantially increased AUC.